# TRAINER

In [26]:
import re
import torch
import torch.cuda

import evaluate
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import word_tokenize

from PIL import Image
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration
from transformers import BitsAndBytesConfig

In [30]:
# Activate GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

# MULTIMODAL LABELS
columns = ["uri","text","Additiva","Amore","Anticipazione","Citazione","Contenitore","Contenuto","Disgusto","Divergente","Fiducia","Gioia","Hate speech","Incomprensibile","Ironia","Neutro","Offensivita","Parallela","Paura","Rabbia","Sarcasmo","Sorpresa","Tristezza","img","img_path"]

to_keep=["text","Additiva","Amore","Anticipazione","Disgusto","Divergente","Fiducia","Gioia","Neutro","Parallela","Paura","Rabbia","Sorpresa","Tristezza","img_path"]
target_names = ['Additiva','Divergente','Parallela']

# DATASET

In [31]:
def split_dataset(data, test_size=0.2, eval_size=0.2, random_state=None):
    train_data, test_data = train_test_split(
        data, test_size=test_size, random_state=random_state
    )
    
    adjusted_eval_size = eval_size / (1 - test_size)
    
    train_data, eval_data = train_test_split(
        train_data, test_size=adjusted_eval_size, random_state=random_state
    )
    print(f"Train size: {len(train_data)}")
    print(f"Test size: {len(test_data)}")
    print(f"Eval size: {len(eval_data)}")
    return train_data, test_data, eval_data

# Example usage


In [52]:
df = pd.read_csv('../datasets/final_multimodal_hot.csv')
# df = pd.read_csv('../datasets/emotion_target_hate_irony_prompted.csv')

df = df[to_keep]

df.drop(['Amore','Anticipazione','Disgusto','Fiducia','Gioia','Neutro','Paura','Rabbia','Sorpresa','Tristezza'], axis=1, inplace=True)

train_df, test_df, eval_df = split_dataset(df, test_size=0.2, eval_size=0.2, random_state=42) 

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
eval_df.reset_index(drop=True, inplace=True)

not_chosen_columns = ["text", "img_path"]
labels = ['Additive', 'Divergent', 'Parallel']

id2label = {idx:label for idx, label in enumerate(labels)}
label2id = {label:idx for idx, label in enumerate(labels)}

Train size: 387
Test size: 129
Eval size: 129


## Preparazione Immagini e output desiderato

In [58]:
import os
from PIL import Image
import requests
from io import BytesIO
import numpy as np
from torchvision.transforms import ToPILImage


def load_image_from_url_or_path(local_path):
    if os.path.isfile(local_path):
        # Local path
        try:
            image = Image.open(local_path)
            return image
        except Exception as e:
            print(f"Unable to load image from path: {e}")
            return None
    else:
        # URL path
        try:
            response = requests.get(path_or_url)
            response.raise_for_status()
            image = Image.open(BytesIO(response.content))
            return image
        except Exception as e:
            print(f"Unable to load image from URL: {e}")
            return None

def preprocess_labels(examples):
    # Handle a single row
    text = examples["text"]
    image_folder = "../images"

    path = os.path.join(image_folder, examples["img_path"])
    examples["img_path"] = path
    # Create labels matrix for a single example
    labels_batch = {k: examples[k] for k in examples.keys() if k in target_names}
    labels_matrix = np.zeros((1, len(target_names)))
    
    for idx, label in enumerate(target_names):
        labels_matrix[0, idx] = labels_batch[label]

    labels_matrix = labels_matrix.astype(int)

    # Add processed fields to the row
    examples["labels"] = labels_matrix[0]
    examples["out"] = f"Relationship detected: {''.join([id2label[i] for i, val in enumerate(labels_matrix[0]) if val == 1])}"

    examples["image"] = load_image_from_url_or_path(path)

    return examples

In [60]:
train_df = train_df.apply(preprocess_labels, axis=1)
test_df = test_df.apply(preprocess_labels, axis=1)
eval_df = eval_df.apply(preprocess_labels, axis=1)

In [61]:
from torch.utils.data import Dataset

class PreloadedDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = row["image"]
        if self.transform:
            image = self.transform(image)
        text = row["text"]
        out = row["out"]
        return {"image": image, "text": text, "out": out}

# Example usage
from torchvision import transforms

train_dataset = PreloadedDataset(train_df)
test_dataset = PreloadedDataset(test_df)
eval_dataset = PreloadedDataset(eval_df)

# LOAD MODEL

In [11]:
model_id = "mistral-community/pixtral-12b"

In [12]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(model_id,
                                                      torch_dtype=torch.float16,
                                                      device_map=device,
                                                      quantization_config=quantization_config)

RuntimeError: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
No module named 'triton.ops'

In [ ]:
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.add_special_tokens({'pad_token': '<pad>'})
# model.resize_token_embeddings(len(processor.tokenizer))

## Prompt Normale

In [ ]:
sys = (
    "You are a bot expert in Multimodal Classification. You will classify the relationship between a text and an image posted together on social media, using only the following categories:"
    " Additive (the text and the image enrich each other, combining to create a deeper or amplified meaning),"
    " Parallel (the text and the image independently express the same message or sentiment),"
    " or Divergent (the text and the image express different or conflicting meanings, often producing irony or non sense)."

    " You will be given a Twitter post in Italian consisting of a text (caption or comment) and an accompanying image."
    " Consider how the image and text work together: do they support each other, extend each other's meaning, or clash to form a new interpretation?"
    " Your task is to return only ONE label: Additive, Parallel, or Divergent."

    " The Twitter posts are related to two Italian TV programs:"
    " 'Pechino Express', a reality-adventure show where pairs travel with little money, complete challenges, and explore cultures through local hospitality;"
    " and 'Propaganda Live', a satirical talk show blending politics, field reports, current events, live music, and debates with various guests."
    
    "Keep in mind:"
    "- If both text and image express similar ideas clearly and directly, the relationship is likely Parallel."
    "- If the image and text combine to create new or expanded meaning, the relation is Additive."
    "- If the image and text conflict in tone, meaning, or intent, the relation is Divergent."
)

usr = (
    "What is the most likely relationship between the image and the text in the following Twitter post?"
    " Choose only one from the following categories: Additive, Parallel, or Divergent."
    " Consider that the text serves as a caption or comment on the image, and reflect on how the two elements interact to convey meaning."
    "\nOutput template:"
    "\nRelationship detected: [Additive / Divergent / Parallel]"
    "\nHere is the text and image to analyze:\n"
)


## Prompt Correlazioni

In [ ]:
sys = (
        "You are a bot expert in Multimodal Classification. You will classify the relationship between a text and an image posted together on social media, using only the following categories:"
    " Additive (the text and the image enrich each other, combining to create a deeper or amplified meaning),"
    " Parallel (the text and the image independently express the same message or sentiment),"
    " or Divergent (the text and the image express different or conflicting meanings, often producing irony or non sense)."

    " You will be given a Twitter post in Italian consisting of a text (caption or comment) and an accompanying image."
    " Consider how the image and text work together: do they support each other, extend each other's meaning, or clash to form a new interpretation?"
    " Your task is to return only ONE label: Additive, Parallel, or Divergent."

    " The Twitter posts are related to two Italian TV programs:"
    " 'Pechino Express', a reality-adventure show where pairs travel with little money, complete challenges, and explore cultures through local hospitality;"
    " and 'Propaganda Live', a satirical talk show blending politics, field reports, current events, live music, and debates with various guests."

    "The sentences to analyze are always preceded with some information regarding other levels of annotation like emotions, irony, hate and target (if it's talking about the content of the show 'Contenuto' or the direction of the show 'Contenitore'."
    "Here are key correlations that can help guide your analysis, based on observed patterns:"
    "- **Additive** relations frequently co-occur with: Anticipation, Trust, Anger, Sadness, and Love."
    "- **Divergent** relations often involve: Joy (ironically), Surprise, Disgust, and Irony."
    "- **Parallel** relations are rare but may involve: Trust, Neutrality."

    "Keep in mind:"
    "- Irony is strongly associated with Divergent relationships."
    "- Joy can occur both genuinely (Additive) and ironically (Divergent), so context matters."
    "- If both text and image express similar ideas clearly and directly, the relationship is likely Parallel."
    "- If the image and text combine to create new or expanded meaning, the relation is Additive."
    "- If the image and text conflict in tone, meaning, or intent, the relation is Divergent."

    "Return only ONE label: Additive, Parallel, or Divergent."
)

usr = (
    "Given the following Twitter post, what is the most likely relationship between the image and the text?"
    " Consider the emotions already detected, and use them to guide your analysis."
    "\nOutput template:"
    "\nRelationship detected: [Additive / Divergent / Parallel]"
)

# DATACOLLATOR

In [ ]:
from PIL import Image

def convert_to_rgb(image):
    if image.mode != "RGB":
        return image.convert("RGB")
    return image

class MyDataCollator:
    
    def __init__(self, processor):
        self.processor = processor
       
    def find_subsequence(self, sequence, subsequence):
            seq_len = len(sequence)
            sub_len = len(subsequence)
            
            for i in range(seq_len - sub_len + 1):
                if torch.equal(sequence[i:i+sub_len], subsequence):
                    return i
            return None
    
    def __call__(self, examples):
        texts=[]
        images=[]
        assistant_res = []

        sys = (
            "You are a bot expert in Multimodal Classification. You will classify the relationship between a text and an image posted together on social media, using only the following categories:"
            " Additive (the text and the image enrich each other, combining to create a deeper or amplified meaning),"
            " Parallel (the text and the image independently express the same message or sentiment),"
            " or Divergent (the text and the image express different or conflicting meanings, often producing irony or non sense)."
        
            " You will be given a Twitter post in Italian consisting of a text (caption or comment) and an accompanying image."
            " Consider how the image and text work together: do they support each other, extend each other's meaning, or clash to form a new interpretation?"
            " Your task is to return only ONE label: Additive, Parallel, or Divergent."
        
            " The Twitter posts are related to two Italian TV programs:"
            " 'Pechino Express', a reality-adventure show where pairs travel with little money, complete challenges, and explore cultures through local hospitality;"
            " and 'Propaganda Live', a satirical talk show blending politics, field reports, current events, live music, and debates with various guests."
            
            "Keep in mind:"
            "- If both text and image express similar ideas clearly and directly, the relationship is likely Parallel."
            "- If the image and text combine to create new or expanded meaning, the relation is Additive."
            "- If the image and text conflict in tone, meaning, or intent, the relation is Divergent."
        )
        
        usr = (
            "What is the most likely relationship between the image and the text in the following Twitter post?"
            " Choose only one from the following categories: Additive, Parallel, or Divergent."
            " Consider that the text serves as a caption or comment on the image, and reflect on how the two elements interact to convey meaning."
            "\nOutput template:"
            " \n[Explain the reasoning]"
            " \nRelationship detected: [Additive / Divergent / Parallel]"
            "\nHere is the text and image to analyze:\n"
        )
        
        for example in examples:
            image = example["image"]
            image = convert_to_rgb(image)
            question = usr + example['text']
            answer = example['out']
            messages = [
                    {
                        "role": "system",
                        "content": sys,
                    },
                    {
                        "role": "user",
                        "content": [{"type": "text", "content": question},{"type": "image"} ],
                    },
                    {
                        "role": "assistant",
                        "content": answer, 
                    },
                ]
            
            text = self.processor.apply_chat_template(messages)
            
            texts.append(text.strip())
            images.append([image])
            assistant_res.append(answer)
            
            
        batch = self.processor(text = texts, images = images, return_tensors="pt", padding=True)
        
        labels = batch["input_ids"].clone()
        
        for i, (input_ids, assistant_response) in enumerate(zip(batch["input_ids"], assistant_res)):
            
            assistant_tokens = self.processor.tokenizer(assistant_response, return_tensors ="pt")['input_ids'].squeeze(0).to(input_ids.dtype)
            start_idx = self.find_subsequence(input_ids, assistant_tokens)
            
            if start_idx is not None:
                labels[i, :start_idx] = -100
                labels[i, start_idx+len(assistant_tokens):] = -100
                
        batch["labels"] = labels
        return batch
        

data_collator = MyDataCollator(processor)

## Lora

In [ ]:
from peft import LoraConfig
from peft import get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    use_rslora=True,
    target_modules="all-linear",
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
    
)

model = get_peft_model(model,lora_config).to(device)

In [ ]:
model.print_trainable_parameters()

## Training

In [ ]:
from transformers import TrainingArguments, Trainer

epochs = 2
lr = 3e-5
schedule="linear"

run_name = f"pixtral_finetune-{epochs}_epochs_{lr}_lr"

training_args = TrainingArguments(
    num_train_epochs=epochs,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=lr,
    weight_decay=0.01,
    logging_steps=0.1,
    output_dir="fine-tuned_model",
    evaluation_strategy="epoch",
    do_eval=True, 
    save_strategy="epoch", 
    lr_scheduler_type=schedule,
    fp16=True,
    hub_model_id="lupobricco/prosat_multimodal_pixtral",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [ ]:
trainer.train()

# Save model da cambiare il path

In [ ]:
trainer.save_model(path)